In [ ]:
import numpy as np
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import sampo.scheduler

from sampo.userinput.parser.csv_parser import CSVParser
from sampo.hybrid.cycle import CycleHybridScheduler
from sampo.api.genetic_api import ScheduleGenerationScheme, Individual
from sampo.scheduler import HEFTScheduler, HEFTBetweenScheduler, TopologicalScheduler, GeneticScheduler
from sampo.hybrid.population import HeuristicPopulationScheduler, GeneticPopulationScheduler

from sampo.generator.environment import get_contractor_by_wg
from sampo.generator import SimpleSynthetic

from sampo.base import SAMPO
from sampo.schemas import WorkGraph
from sampo.schemas.time import Time

# Импорт ресурсной модели
from models.field_dev_resources_time_estimator import FieldDevWorkEstimator

# Фитнесс-функции
from sampo.scheduler.genetic.operators import TimeFitness, DeadlineCostFitness

# Потом можно заменить, если добавить в функциях аргументы по умолчанию
from sampo.schemas.landscape import LandscapeConfiguration
from sampo.schemas.schedule_spec import ScheduleSpec

# ??
from my_utils.precedence import PrecedenceManager
from my_utils.project_info import ProjectInfo

# Визуализация
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
def run_experiment(wg, contractors):

    heuristics = HeuristicPopulationScheduler([
        HEFTScheduler(work_estimator=project_work_estimator), 
        HEFTBetweenScheduler(work_estimator=project_work_estimator), 
        TopologicalScheduler(work_estimator=project_work_estimator)
    ])
    # genetic1 = TabuPopulationScheduler()
    genetic1 = GeneticPopulationScheduler(GeneticScheduler(mutate_order=0.2,
                                                           mutate_resources=0.2,
                                                           sgs_type=ScheduleGenerationScheme.Parallel,
                                                           work_estimator=project_work_estimator))
    genetic2 = GeneticPopulationScheduler(GeneticScheduler(mutate_order=0.001,
                                                           mutate_resources=0.001,
                                                           sgs_type=ScheduleGenerationScheme.Parallel,
                                                           work_estimator=project_work_estimator))

    hybrid_combine = CycleHybridScheduler(heuristics, [genetic1, genetic2], max_plateau_size=1)
    hybrid_genetic1 = CycleHybridScheduler(heuristics, [genetic1], max_plateau_size=1)
    hybrid_genetic2 = CycleHybridScheduler(heuristics, [genetic2], max_plateau_size=1)

    # wg = WorkGraph.load('wgs', f'{graph_size}_{iteration}')
    # contractors = [get_contractor_by_wg(wg)]

    # SAMPO.backend = MultiprocessingComputationalBackend(n_cpus=10)
    SAMPO.backend.cache_scheduler_info(wg, contractors, work_estimator=project_work_estimator)
    SAMPO.backend.cache_genetic_info()

    schedule_hybrid_combine = hybrid_combine.schedule(wg, contractors, work_estimator=project_work_estimator)
    schedule_genetic1 = hybrid_genetic1.schedule(wg, contractors, work_estimator=project_work_estimator)
    schedule_genetic2 = hybrid_genetic2.schedule(wg, contractors, work_estimator=project_work_estimator)

    # print(f'Hybrid combine: {schedule_hybrid_combine.execution_time}')
    # print(f'Scheduler 1 cycled: {schedule_genetic1.execution_time}')
    # print(f'Scheduler 2 cycled: {schedule_genetic2.execution_time}')
    return schedule_hybrid_combine.execution_time, schedule_genetic1.execution_time, schedule_genetic2.execution_time

## Get project data

In [ ]:
DATA_PATH = Path('data/csv/gas_network_full_connections.csv')
HISTORY_PATH = Path('data/historical_projects_data.csv')

In [ ]:
project_df = pd.read_csv(DATA_PATH, sep=',')
history_df = pd.read_csv(HISTORY_PATH, sep=';')

In [ ]:
all_connections = True
change_connections = False

In [ ]:
project_work_estimator = FieldDevWorkEstimator()

In [ ]:
%%capture
# not print output

wg, contractors = CSVParser.work_graph_and_contractors(
    works_info=CSVParser.read_graph_info(
        project_info=project_df,
        history_data=history_df,
        all_connections=all_connections,
        change_connections_info=change_connections
    ),
    work_resource_estimator=project_work_estimator
)

## Run

In [ ]:
# results = run_experiment(wg, contractors)

In [ ]:
# results

In [ ]:
heuristics = HeuristicPopulationScheduler([
    HEFTScheduler(work_estimator=project_work_estimator), 
    HEFTBetweenScheduler(work_estimator=project_work_estimator), 
    TopologicalScheduler(work_estimator=project_work_estimator)
])

genetic1 = GeneticPopulationScheduler(GeneticScheduler(mutate_order=0.2,
                                                       mutate_resources=0.2,
                                                       sgs_type=ScheduleGenerationScheme.Parallel,
                                                       work_estimator=project_work_estimator))

hybrid_genetic1 = CycleHybridScheduler(heuristics, [genetic1], max_plateau_size=1)

In [ ]:
SAMPO.backend.cache_scheduler_info(wg, contractors, work_estimator=project_work_estimator)
SAMPO.backend.cache_genetic_info()

population = hybrid_genetic1.run(wg, contractors)

In [ ]:
[i.fitness.values[0] for i in population]

In [ ]:
# Individual(individual_fitness_constructor=TimeFitness, chromosome=population[0][:])

In [ ]:
# ind = Individual(individual_fitness_constructor=TimeFitness, chromosome=random_population[0])

In [ ]:
population[0]

## Test backend

In [ ]:
project_info = ProjectInfo(wg, contractors)
precedence_manager = PrecedenceManager(project_info.children, project_info.parents)

In [ ]:
def get_random_chromosome():

    activity_list = precedence_manager.get_activity_list(np.random.random(project_info.n_works))
    
    resource_amounts = np.zeros((project_info.n_works, project_info.n_workers), dtype=np.int16)
    for work_id in range(project_info.n_works):
        for worker_id in range(project_info.n_workers):
            min_amount, max_amount = project_info.get_resource_borders(work_id=work_id, worker_id=worker_id)
            percent = np.random.random()
            chosen_amount = min_amount + (max_amount - min_amount) * percent
            resource_amounts[work_id, worker_id] = int(chosen_amount)
    
    ceil_list = project_info.contractor_borders
    
    contractor_choices_list = np.zeros(project_info.n_works, dtype=np.int16)
    resource_amounts = np.append(resource_amounts, contractor_choices_list.reshape(-1, 1), axis=1)
    
    # TODO do we need it?
    spec = ScheduleSpec()
    zones = np.array([])
    
    chromosome = (activity_list, resource_amounts, ceil_list, spec, zones)
    
    project_info.assert_chromosome_correct(chromosome)

    return chromosome

In [ ]:
project_info.assert_chromosome_correct(get_random_chromosome())

In [ ]:
# random_population = [get_random_chromosome() for _ in range(100)]

In [ ]:
SAMPO.backend.cache_scheduler_info(wg, contractors, work_estimator=project_work_estimator)
SAMPO.backend.cache_genetic_info()

In [ ]:
# SAMPO.backend.compute_chromosomes(DeadlineCostFitness(deadline=Time(700)), random_population)[:5]

In [ ]:
# np.min([i[0] for i in SAMPO.backend.compute_chromosomes(DeadlineCostFitness(deadline=Time(700)), random_population)])

In [ ]:
SAMPO.backend.compute_chromosomes(DeadlineCostFitness(deadline=Time(700)), [random_population[0]])

## Simulated Annealing

In [ ]:
population = [Individual(individual_fitness_constructor=TimeFitness, chromosome=get_random_chromosome()) for _ in range(50)]
fitnesses = SAMPO.backend.compute_chromosomes(DeadlineCostFitness(deadline=Time(700)), population)
for i, fitness in enumerate(fitnesses):
    population[i].fitness.values = fitness

In [ ]:
np.mean([i[0] for i in fitnesses])

In [ ]:
def current_single_fitness_function(activity_list, resources_matrix, ceil_list):
    full_chromosome = [activity_list, resources_matrix, ceil_list, ScheduleSpec(), np.array([])]
    project_info.assert_chromosome_correct(full_chromosome)
    fitness = SAMPO.backend.compute_chromosomes(DeadlineCostFitness(deadline=Time(700)), [full_chromosome])
    return fitness[0][0]

In [ ]:
class Solution:

    def __init__(self, activity_list, resources_matrix, ceil_list, fitness=None):
        self.activity_list = np.array(activity_list).copy()
        self.resources_matrix = resources_matrix.copy()
        self.ceil_list = ceil_list.copy()
        if fitness is None:
            self.fitness = current_single_fitness_function()
        else:
            self.fitness = fitness

    @classmethod
    def from_individual(cls, individual):
        return cls(
            activity_list=individual[0], 
            resources_matrix=individual[1], 
            ceil_list=individual[2], 
            fitness=individual.fitness.values[0]
        )
    
    def jump(self):

        new_activity_list = self.activity_list.copy()
        new_resources_matrix = self.resources_matrix.copy()
        
        n_resource_mutations = np.random.randint(0, 3+1)
        for _ in range(n_resource_mutations):
            work_to_mutate, worker_to_mutate = project_info.nonzero_resources_indices[np.random.choice(len(project_info.nonzero_resources_indices))]
            min_amount, max_amount = project_info.get_resource_borders(work_id=work_to_mutate, worker_id=worker_to_mutate)
            new_resources_matrix[work_to_mutate, worker_to_mutate] = np.random.randint(min_amount, max_amount+1)
    
        n_order_mutations = np.random.randint(0, 3+1)
        for _ in range(n_order_mutations):
            a, b = np.random.randint(project_info.n_works, size=2)
            new_activity_list[[a, b]] = new_activity_list[[b, a]]
        new_activity_list = precedence_manager.convert_al_to_valid_order(new_activity_list)

        new_fitness = current_single_fitness_function(new_activity_list, new_resources_matrix, self.ceil_list)       
        delta_ratio = new_fitness / self.fitness - 1
        temperature = 0.02
        
        if (new_fitness < self.fitness) or (np.random.random() < np.exp(-delta_ratio/temperature)):
            self.activity_list = np.array(new_activity_list).copy()
            self.resources_matrix = new_resources_matrix.copy()
            self.fitness = new_fitness                

In [ ]:
a = Solution.from_individual(population[3])

In [ ]:
history = []
for i in range(2000):
    a.jump()
    history.append(a.fitness)

In [ ]:
fig = px.line(history, template='plotly_white')
fig.show()

In [ ]:
np.min(history)

## Bee colony

In [ ]:
class Bee:

    def __init__(self, activity_list, resources_matrix, ceil_list, fitness=None):
        self.activity_list = np.array(activity_list).copy()
        self.resources_matrix = resources_matrix.copy()
        self.ceil_list = ceil_list.copy()
        if fitness is None:
            self.fitness = current_single_fitness_function()
        else:
            self.fitness = fitness
        self.n_tries = 0

    @classmethod
    def from_individual(cls, individual):
        return cls(
            activity_list=individual[0], 
            resources_matrix=individual[1], 
            ceil_list=individual[2], 
            fitness=individual.fitness.values[0]
        )
    
    def reset(self):
        self.activity_list = np.array(get_random_chromosome()[0])
        self.resources_matrix = get_random_chromosome()[1]
        self.fitness = current_single_fitness_function(self.activity_list, self.resources_matrix, self.ceil_list)     
        self.n_tries = 0

    def random_mutation(self):

        new_activity_list = self.activity_list.copy()
        new_resources_matrix = self.resources_matrix.copy()
        
        n_resource_mutations = np.random.randint(0, 10+1)
        for _ in range(n_resource_mutations):
            work_to_mutate, worker_to_mutate = project_info.nonzero_resources_indices[np.random.choice(len(project_info.nonzero_resources_indices))]
            min_amount, max_amount = project_info.get_resource_borders(work_id=work_to_mutate, worker_id=worker_to_mutate)
            new_resources_matrix[work_to_mutate, worker_to_mutate] = np.random.randint(min_amount, max_amount+1)
    
        n_order_mutations = np.random.randint(0, 10+1)
        for _ in range(n_order_mutations):
            a, b = np.random.randint(project_info.n_works, size=2)
            new_activity_list[[a, b]] = new_activity_list[[b, a]]
        new_activity_list = precedence_manager.convert_al_to_valid_order(new_activity_list)

        new_fitness = current_single_fitness_function(new_activity_list, new_resources_matrix, self.ceil_list)       
        
        if new_fitness < self.fitness:
            self.activity_list = np.array(new_activity_list).copy()
            self.resources_matrix = new_resources_matrix.copy()
            self.fitness = new_fitness
        else:
            self.n_tries += 1
            if self.n_tries >= 50:
                self.reset()

    def towards_mutation(self, other):

        # TODO how to do a "little" crossover??
        new_activity_list = self.activity_list.copy()
        # new_activity_list = np.where(
        #     np.random.choice([0, 1], p=[0.90, 0.10], size=project_info.n_works),
        #     self.activity_list,
        #     other.activity_list
        # )
        
        new_resources_matrix = np.where(
            np.random.choice([0, 1], p=[0.90, 0.10], size=(project_info.n_works, project_info.n_workers+1)),
            self.resources_matrix,
            other.resources_matrix
        )

        new_fitness = current_single_fitness_function(new_activity_list, new_resources_matrix, self.ceil_list)       
        
        if new_fitness < self.fitness:
            self.activity_list = np.array(new_activity_list).copy()
            self.resources_matrix = new_resources_matrix.copy()
            self.fitness = new_fitness
        else:
            self.n_tries += 1
            if self.n_tries >= 50:
                self.reset()

In [ ]:
class BeeColony:

    def __init__(self, bees):
        self.bees = bees
        self.n_bees = len(bees)

    @classmethod
    def from_population(cls, population):
        return cls([Bee.from_individual(i) for i in population])
    
    def mutate(self):
        for _ in range(self.n_bees // 2):
            bee_index = np.random.choice(self.n_bees)
            self.bees[bee_index].random_mutation()
        
        for _ in range(self.n_bees // 2):
            bee_1_index = np.random.choice(self.n_bees)
            bee_2_index = np.random.choice(self.n_bees)
            self.bees[bee_1_index].towards_mutation(self.bees[bee_2_index])

    def get_fitness_stats(self):
        fitness_array = np.array([bee.fitness for bee in self.bees])
        return np.mean(fitness_array), np.min(fitness_array)

In [ ]:
bee_colony = BeeColony.from_population(population)

gen_means = []
gen_mins = []
gen_mean, gen_min = bee_colony.get_fitness_stats()
gen_means.append(gen_mean)
gen_mins.append(gen_min)

for generation_number in range(200):
    print(generation_number, end='\r')
    
    bee_colony.mutate()
    gen_mean, gen_min = bee_colony.get_fitness_stats()
    gen_means.append(gen_mean)
    gen_mins.append(gen_min)

In [ ]:
fig = go.Figure([
    go.Scatter(y=gen_mins, name="Gen Best", line_color="navy"),
    go.Scatter(y=gen_means, name="Gen Mean", line_color="green"),
    # go.Scatter(y=[baseline_random_search for _ in gen_means], name="Baseline", line_color="lightgrey"),
    
])

fig.update_layout(height=500, width=1000, template="plotly_white")
fig.update_layout(
    xaxis_title="Поколение",                   
    yaxis_title="Метрика (стоимость + штраф)",
    showlegend=False,
    margin_t=0, margin_b=0, margin_l=0, margin_r=0
)

fig.show()
# fig.write_image('plots/bees_4.png', scale=2)

In [ ]:
np.min(gen_mins)

## Artificial Immune

In [ ]:
class Antibody:

    def __init__(self, activity_list, resources_matrix, ceil_list, fitness=None):
        self.activity_list = np.array(activity_list).copy()
        self.resources_matrix = resources_matrix.copy()
        self.ceil_list = ceil_list.copy()
        if fitness is None:
            self.fitness = current_single_fitness_function(activity_list, resources_matrix, ceil_list)
        else:
            self.fitness = fitness

    @classmethod
    def from_individual(cls, individual):
        return cls(
            activity_list=individual[0], 
            resources_matrix=individual[1], 
            ceil_list=individual[2], 
            fitness=individual.fitness.values[0]
        )

    @classmethod
    def get_random_new(cls):
        activity_list = np.array(get_random_chromosome()[0])
        resources_matrix = get_random_chromosome()[1]
        ceil_list = get_random_chromosome()[2]
        
        return cls(
            activity_list=activity_list, 
            resources_matrix=resources_matrix, 
            ceil_list=ceil_list, 
            fitness=current_single_fitness_function(activity_list, resources_matrix, ceil_list)
        )

    def random_mutation(self):

        new_activity_list = self.activity_list.copy()
        new_resources_matrix = self.resources_matrix.copy()
        
        n_resource_mutations = np.random.randint(0, 10+1)
        for _ in range(n_resource_mutations):
            work_to_mutate, worker_to_mutate = project_info.nonzero_resources_indices[np.random.choice(len(project_info.nonzero_resources_indices))]
            min_amount, max_amount = project_info.get_resource_borders(work_id=work_to_mutate, worker_id=worker_to_mutate)
            new_resources_matrix[work_to_mutate, worker_to_mutate] = np.random.randint(min_amount, max_amount+1)
    
        n_order_mutations = np.random.randint(0, 10+1)
        for _ in range(n_order_mutations):
            a, b = np.random.randint(project_info.n_works, size=2)
            new_activity_list[[a, b]] = new_activity_list[[b, a]]
        new_activity_list = precedence_manager.convert_al_to_valid_order(new_activity_list)

        return Antibody(new_activity_list, new_resources_matrix, self.ceil_list)

In [ ]:
class ArtificialImmuneSystem:

    def __init__(self, antibodies):
        self.antibodies = antibodies
        self.n_antibodies = len(antibodies)

    @classmethod
    def from_population(cls, population):
        return cls([Antibody.from_individual(i) for i in population])

    def update(self):
        affinity = np.array([i.fitness for i in self.antibodies])
        affinity = (affinity - affinity.min()) / (affinity.max() - affinity.min())
        n_clones = np.floor( 1 + 5*affinity ).astype(np.int16)

        new_population = []
        for this_antibody, this_n_clones in zip(self.antibodies, n_clones):
            for k in range(this_n_clones):
                new_population.append( this_antibody.random_mutation() )

        self.antibodies = sorted(self.antibodies + new_population, key=lambda x: x.fitness)

        # replace with selected
        self.antibodies = self.antibodies[:round(self.n_antibodies*0.8)] + list(np.random.choice(
            self.antibodies[round(self.n_antibodies*0.8):], 
            size=round(self.n_antibodies*0.2), 
            replace=False
        ))
        
        # replace with random
        # self.antibodies = self.antibodies[:round(self.n_antibodies*0.8)] + [Antibody.get_random_new() for _ in range(round(self.n_antibodies*0.2))]

    def get_fitness_stats(self):
        fitness_array = np.array([i.fitness for i in self.antibodies])
        return np.mean(fitness_array), np.min(fitness_array)

In [ ]:
immune = ArtificialImmuneSystem.from_population(population)

gen_means = []
gen_mins = []
gen_mean, gen_min = immune.get_fitness_stats()
gen_means.append(gen_mean)
gen_mins.append(gen_min)

for generation_number in range(120):
    print(generation_number, end='\r')
    
    immune.update()
    gen_mean, gen_min = immune.get_fitness_stats()
    gen_means.append(gen_mean)
    gen_mins.append(gen_min)

In [ ]:
fig = go.Figure([
    go.Scatter(y=gen_mins, name="Gen Best", line_color="navy"),
    go.Scatter(y=gen_means, name="Gen Mean", line_color="green"),
    # go.Scatter(y=[baseline_random_search for _ in gen_means], name="Baseline", line_color="lightgrey"),
    
])

fig.update_layout(height=500, width=1000, template="plotly_white")
fig.update_layout(
    xaxis_title="Поколение",                   
    yaxis_title="Метрика (стоимость + штраф)",
    showlegend=False,
    margin_t=0, margin_b=0, margin_l=0, margin_r=0
)

fig.show()
# fig.write_image('plots/immune_1.png', scale=2)

In [ ]:
np.min(gen_mins)